# Colab Session: thai-sign-train
Generated from colab-cli history log.

**Session Created**: 2026-06-19 12:19:35
- Endpoint: `gpu-t4-s-kkb-usw1b2-j29pchcds6jw`
- Hardware: `T4`

### Automation: drivemount (2026-06-19 12:19:35)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**Session Created**: 2026-06-19 12:30:05
- Endpoint: `gpu-t4-s-kkb-usw1b1-26cistbk0gmw1`
- Hardware: `T4`

*File Operation*: `upload` on `/content/thai-sign-code.zip`

*File Operation*: `upload` on `/content/tsl51_v3.zip`

*File Operation*: `ls` on `/content`

*File Operation*: `upload` on `/content/youtube_sl25_thai_v3.zip.part003`

*File Operation*: `upload` on `/content/kaggle.json`

*File Operation*: `upload` on `/content/start_colab_training.py`

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import stat
import subprocess
import sys
import zipfile
from pathlib import Path

REPO_ZIP = Path('/content/thai-sign-code.zip')
REPO_ROOT = Path('/content/thai-sign-translator')
KAGGLE_SRC = Path('/content/kaggle.json')
KAGGLE_HOME = Path('/root/.kaggle')
KAGGLE_DST = KAGGLE_HOME / 'kaggle.json'
DATA_BASE = Path('/content/kaggle')
DATASETS = {
    'tsl51_v3': 'orbitorls/thai-sign-tsl51',
    'youtube_sl25_thai_v3': 'orbitorls/thai-sign-youtube',
}
OUT_DIR = Path('/content/checkpoints/pose_t5_v3_colab')
LOG_PATH = OUT_DIR / 'train.log'
PID_PATH = OUT_DIR / 'train.pid'
META_PATH = OUT_DIR / 'launch.json'


def ensure_repo() -> None:
    REPO_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(REPO_ZIP, 'r') as archive:
        archive.extractall(REPO_ROOT)


def ensure_kaggle_auth() -> None:
    if not KAGGLE_SRC.is_file():
        raise FileNotFoundError('Missing /content/kaggle.json for Kaggle authentication')
    KAGGLE_HOME.mkdir(parents=True, exist_ok=True)
    shutil.copy2(KAGGLE_SRC, KAGGLE_DST)
    os.chmod(KAGGLE_DST, stat.S_IRUSR | stat.S_IWUSR)


def ensure_kaggle_dataset(folder_name: str, slug: str) -> Path:
    target = DATA_BASE / folder_name
    manifest = target / 'manifest.csv'
    if manifest.is_file():
        return target
    target.mkdir(parents=True, exist_ok=True)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'])
    subprocess.check_call(['kaggle', 'datasets', 'download', '-d', slug, '-p', str(target), '--unzip'])
    if not manifest.is_file():
        raise FileNotFoundError(f'Manifest not found after downloading {slug}: {manifest}')
    return target


def main() -> None:
    ensure_repo()
    ensure_kaggle_auth()
    data_roots = [str(ensure_kaggle_dataset(name, slug)) for name, slug in DATASETS.items()]
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    cmd = [
        sys.executable,
        '-u',
        str(REPO_ROOT / 'scripts' / 'colab_train_pose_t5.py'),
        '--repo-zip', str(REPO_ZIP),
        '--repo-root', str(REPO_ROOT),
        '--data-roots', ','.join(data_roots),
        '--out-dir', str(OUT_DIR),
        '--lr', '1e-4',
        '--dropout', '0.3',
        '--weight-decay', '0.05',
        '--early-stopping-patience', '10',
        '--early-stopping-min-delta', '0.0',
        '--eval-steps', '100',
        '--batch-size', '8',
        '--grad-accum', '4',
        '--resume', 'auto',
        '--amp', 'auto',
        '--max-runtime-min', '690',
    ]

    with LOG_PATH.open('ab', buffering=0) as handle:
        handle.write(b'[launcher] starting background training\n')
        proc = subprocess.Popen(
            cmd,
            cwd=str(REPO_ROOT),
            stdout=handle,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )

    PID_PATH.write_text(str(proc.pid), encoding='utf-8')
    META_PATH.write_text(json.dumps({'pid': proc.pid, 'cmd': cmd, 'data_roots': data_roots}, indent=2), encoding='utf-8')
    print(json.dumps({'status': 'started', 'pid': proc.pid, 'out_dir': str(OUT_DIR), 'data_roots': data_roots}))


if __name__ == '__main__':
    main()


CalledProcessError: Command '['kaggle', 'datasets', 'download', '-d', 'orbitorls/thai-sign-tsl51', '-p', '/content/kaggle/tsl51_v3', '--unzip']' returned non-zero exit status 1.

In [ ]:
from __future__ import annotations
import json
import os
import shutil
import stat
import subprocess
from pathlib import Path

src = Path('/content/kaggle.json')
dst_dir = Path('/root/.kaggle')
dst = dst_dir / 'kaggle.json'
dst_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(src, dst)
os.chmod(dst, stat.S_IRUSR | stat.S_IWUSR)
subprocess.run(['python', '-m', 'pip', 'install', '-q', 'kaggle'], check=True)
for cmd in [
    ['kaggle', 'config', 'view'],
    ['kaggle', 'datasets', 'files', '-d', 'orbitorls/thai-sign-tsl51'],
    ['kaggle', 'datasets', 'download', '-d', 'orbitorls/thai-sign-tsl51', '-p', '/content/kaggle/tsl51_v3', '--unzip'],
]:
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(json.dumps({'cmd': cmd, 'returncode': result.returncode, 'stdout': result.stdout[-4000:], 'stderr': result.stderr[-4000:]}, ensure_ascii=False))


{"cmd": ["kaggle", "config", "view"], "returncode": 0, "stdout": "Configuration values from /root/.kaggle\n- username: pannawat.jjk@gmail.com\n- auth_method: LEGACY_API_KEY\n- path: None\n- proxy: None\n- competition: None\n", "stderr": ""}


{"cmd": ["kaggle", "datasets", "files", "-d", "orbitorls/thai-sign-tsl51"], "returncode": 1, "stdout": "403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/ListDatasetFiles\n", "stderr": ""}


{"cmd": ["kaggle", "datasets", "download", "-d", "orbitorls/thai-sign-tsl51", "-p", "/content/kaggle/tsl51_v3", "--unzip"], "returncode": 1, "stdout": "403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDatasetMetadata\n", "stderr": ""}


*File Operation*: `upload` on `/content/access_token`

In [ ]:
from __future__ import annotations
import json
import os
import shutil
import stat
import subprocess
from pathlib import Path

src = Path('/content/access_token')
dst_dir = Path('/root/.kaggle')
dst = dst_dir / 'access_token'
dst_dir.mkdir(parents=True, exist_ok=True)
if (dst_dir / 'kaggle.json').exists():
    (dst_dir / 'kaggle.json').unlink()
shutil.copy2(src, dst)
os.chmod(dst, stat.S_IRUSR | stat.S_IWUSR)
subprocess.run(['python', '-m', 'pip', 'install', '-q', 'kaggle'], check=True)
for cmd in [
    ['kaggle', 'config', 'view'],
    ['kaggle', 'datasets', 'files', '-d', 'orbitorls/thai-sign-tsl51'],
]:
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(json.dumps({'cmd': cmd, 'returncode': result.returncode, 'stdout': result.stdout[-4000:], 'stderr': result.stderr[-4000:]}, ensure_ascii=False))


{"cmd": ["kaggle", "config", "view"], "returncode": 0, "stdout": "Warning: Looks like you're using an outdated `kaggle` version (installed: 2.0.2), please consider upgrading to the latest version (2.2.2)\nConfiguration values from /root/.kaggle\n- username: orbitorls\n- auth_method: ACCESS_TOKEN\n- path: None\n- proxy: None\n- competition: None\n", "stderr": ""}


{"cmd": ["kaggle", "datasets", "files", "-d", "orbitorls/thai-sign-tsl51"], "returncode": 0, "stdout": "Warning: Looks like you're using an outdated `kaggle` version (installed: 2.0.2), please consider upgrading to the latest version (2.2.2)\nNext Page Token = CfDJ8Ih_VxHsLwdJhHStLnSaLO8CCYinYZNpu5kGBxQPU9fVAu5fUWfH6ml3IgRa4OwNNVfdal3m4FkyhlwIYzfN_dgU5GwvAZhx7SFp-aeTXwjVWX5tJkxLVrnzRfvxxPW5Ut733gPdCN8S-lUpQQ1-19NomNbmaMxUbqgbhjdGtF-F2hgGnggg57MFWVvoQGFcranV2nxQahYv5w6We4FVjx3sQcqkCr5Rr3R87hr3dRBnvJ9OpjzO1_F4NjRlwLY_UBnVS4r0g1ldE2g\nname                                                                            size  creationDate                \n----------------------------------------------------------------------------  ------  --------------------------  \nlandmarks/กรุงเทพ_var_1__ฉัน_var_1__เที่ยว_var_1__ชอบ_var_1_no_space_0.npy    375776  2026-06-16 16:41:02.166000  \nlandmarks/กรุงเทพ_var_1__ฉัน_var_1__เที่ยว_var_1__ชอบ_var_1_no_space_1.npy    375776  2026-06-16 16:41:02.170000  

*File Operation*: `upload` on `/content/start_colab_training.py`

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import stat
import subprocess
import sys
import zipfile
from pathlib import Path

REPO_ZIP = Path('/content/thai-sign-code.zip')
REPO_ROOT = Path('/content/thai-sign-translator')
TOKEN_SRC = Path('/content/access_token')
KAGGLE_HOME = Path('/root/.kaggle')
TOKEN_DST = KAGGLE_HOME / 'access_token'
DATA_BASE = Path('/content/kaggle')
DATASETS = {
    'tsl51_v3': 'orbitorls/thai-sign-tsl51',
    'youtube_sl25_thai_v3': 'orbitorls/thai-sign-youtube',
}
OUT_DIR = Path('/content/checkpoints/pose_t5_v3_colab')
LOG_PATH = OUT_DIR / 'train.log'
PID_PATH = OUT_DIR / 'train.pid'
META_PATH = OUT_DIR / 'launch.json'


def ensure_repo() -> None:
    REPO_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(REPO_ZIP, 'r') as archive:
        archive.extractall(REPO_ROOT)


def ensure_kaggle_auth() -> None:
    if not TOKEN_SRC.is_file():
        raise FileNotFoundError('Missing /content/access_token for Kaggle authentication')
    KAGGLE_HOME.mkdir(parents=True, exist_ok=True)
    shutil.copy2(TOKEN_SRC, TOKEN_DST)
    os.chmod(TOKEN_DST, stat.S_IRUSR | stat.S_IWUSR)
    legacy = KAGGLE_HOME / 'kaggle.json'
    if legacy.exists():
        legacy.unlink()


def ensure_kaggle_dataset(folder_name: str, slug: str) -> Path:
    target = DATA_BASE / folder_name
    manifest = target / 'manifest.csv'
    if manifest.is_file():
        return target
    target.mkdir(parents=True, exist_ok=True)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle>=2.2.2'])
    subprocess.check_call(['kaggle', 'datasets', 'download', '-d', slug, '-p', str(target), '--unzip'])
    if not manifest.is_file():
        raise FileNotFoundError(f'Manifest not found after downloading {slug}: {manifest}')
    return target


def main() -> None:
    ensure_repo()
    ensure_kaggle_auth()
    data_roots = [str(ensure_kaggle_dataset(name, slug)) for name, slug in DATASETS.items()]
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    cmd = [
        sys.executable,
        '-u',
        str(REPO_ROOT / 'scripts' / 'colab_train_pose_t5.py'),
        '--repo-zip', str(REPO_ZIP),
        '--repo-root', str(REPO_ROOT),
        '--data-roots', ','.join(data_roots),
        '--out-dir', str(OUT_DIR),
        '--lr', '1e-4',
        '--dropout', '0.3',
        '--weight-decay', '0.05',
        '--early-stopping-patience', '10',
        '--early-stopping-min-delta', '0.0',
        '--eval-steps', '100',
        '--batch-size', '8',
        '--grad-accum', '4',
        '--resume', 'auto',
        '--amp', 'auto',
        '--max-runtime-min', '690',
    ]

    with LOG_PATH.open('ab', buffering=0) as handle:
        handle.write(b'[launcher] starting background training\n')
        proc = subprocess.Popen(
            cmd,
            cwd=str(REPO_ROOT),
            stdout=handle,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )

    PID_PATH.write_text(str(proc.pid), encoding='utf-8')
    META_PATH.write_text(json.dumps({'pid': proc.pid, 'cmd': cmd, 'data_roots': data_roots}, indent=2), encoding='utf-8')
    print(json.dumps({'status': 'started', 'pid': proc.pid, 'out_dir': str(OUT_DIR), 'data_roots': data_roots}))


if __name__ == '__main__':
    main()


{"status": "started", "pid": 4701, "out_dir": "/content/checkpoints/pose_t5_v3_colab", "data_roots": ["/content/kaggle/tsl51_v3", "/content/kaggle/youtube_sl25_thai_v3"]}


In [ ]:
from __future__ import annotations
import json
import os
from pathlib import Path

out_dir = Path('/content/checkpoints/pose_t5_v3_colab')
pid_path = out_dir / 'train.pid'
log_path = out_dir / 'train.log'
status = {'out_dir_exists': out_dir.exists(), 'pid_path_exists': pid_path.exists(), 'log_path_exists': log_path.exists()}
if pid_path.exists():
    pid = int(pid_path.read_text(encoding='utf-8').strip())
    status['pid'] = pid
    status['proc_exists'] = Path(f'/proc/{pid}').exists()
if log_path.exists():
    text = log_path.read_text(encoding='utf-8', errors='replace')
    status['log_tail'] = text[-3000:]
print(json.dumps(status, ensure_ascii=False))


{"out_dir_exists": true, "pid_path_exists": true, "log_path_exists": true, "pid": 4701, "proc_exists": true, "log_tail": "[launcher] starting background training\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 6.5 MB/s eta 0:00:00\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 89.9 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 86.2 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.2 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 13.8 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.6 MB/s eta 0:00:00\n"}


In [ ]:
from __future__ import annotations
import json
import os
from pathlib import Path

out_dir = Path('/content/checkpoints/pose_t5_v3_colab')
pid_path = out_dir / 'train.pid'
log_path = out_dir / 'train.log'
status = {'out_dir_exists': out_dir.exists(), 'pid_path_exists': pid_path.exists(), 'log_path_exists': log_path.exists()}
if pid_path.exists():
    pid = int(pid_path.read_text(encoding='utf-8').strip())
    status['pid'] = pid
    status['proc_exists'] = Path(f'/proc/{pid}').exists()
if log_path.exists():
    text = log_path.read_text(encoding='utf-8', errors='replace')
    status['log_tail'] = text[-3000:]
print(json.dumps(status, ensure_ascii=False))


{"out_dir_exists": true, "pid_path_exists": true, "log_path_exists": true, "pid": 4701, "proc_exists": true, "log_tail": "[launcher] starting background training\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 6.5 MB/s eta 0:00:00\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 89.9 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 86.2 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.2 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 13.8 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.6 MB/s eta 0:00:00\n[colab] drive_root=/content/drive/MyDrive/thai-sign-translator\n[colab] repo_root=/content/thai-sign-translator\n[colab] data_roots=/content/kaggle/tsl51_v3,/content/kaggle/youtube_sl25_thai_v3\n[colab] out_dir=/content/checkpoints/pose_t5_v3_colab\n[colab] CUDA=True\n

In [ ]:
from __future__ import annotations
import json
import os
from pathlib import Path

out_dir = Path('/content/checkpoints/pose_t5_v3_colab')
pid_path = out_dir / 'train.pid'
log_path = out_dir / 'train.log'
status = {'out_dir_exists': out_dir.exists(), 'pid_path_exists': pid_path.exists(), 'log_path_exists': log_path.exists()}
if pid_path.exists():
    pid = int(pid_path.read_text(encoding='utf-8').strip())
    status['pid'] = pid
    status['proc_exists'] = Path(f'/proc/{pid}').exists()
if log_path.exists():
    text = log_path.read_text(encoding='utf-8', errors='replace')
    status['log_tail'] = text[-3000:]
print(json.dumps(status, ensure_ascii=False))


{"out_dir_exists": true, "pid_path_exists": true, "log_path_exists": true, "pid": 4701, "proc_exists": true, "log_tail": "[launcher] starting background training\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 6.5 MB/s eta 0:00:00\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 89.9 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 86.2 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.2 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 13.8 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.6 MB/s eta 0:00:00\n[colab] drive_root=/content/drive/MyDrive/thai-sign-translator\n[colab] repo_root=/content/thai-sign-translator\n[colab] data_roots=/content/kaggle/tsl51_v3,/content/kaggle/youtube_sl25_thai_v3\n[colab] out_dir=/content/checkpoints/pose_t5_v3_colab\n[colab] CUDA=True\n

*File Operation*: `ls` on `/content/checkpoints/pose_t5_v3_colab`

In [ ]:
from pathlib import Path
p = Path('/content/checkpoints/pose_t5_v3_colab/train.log')
if not p.exists():
    print('NO_LOG')
else:
    text = p.read_text(encoding='utf-8', errors='replace')
    print(text[-8000:])


[launcher] starting background training
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 89.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 86.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.6 MB/s eta 0:00:00
[colab] drive_root=/content/drive/MyDrive/thai-sign-translator
[colab] repo_root=/content/thai-sign-translator
[colab] data_roots=/content/kaggle/tsl51_v3,/content/kaggle/youtube_sl25_thai_v3
[colab] out_dir=/content/checkpoints/pose_t5_v3_colab
[colab] CUDA=True
[colab] GPU=Tesla T4
[colab] VRAM_GB=14.6
2026-06-19 12:41:40.125276: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorF

In [ ]:
from __future__ import annotations
import json
import os
from pathlib import Path

out_dir = Path('/content/checkpoints/pose_t5_v3_colab')
pid_path = out_dir / 'train.pid'
log_path = out_dir / 'train.log'
status = {'out_dir_exists': out_dir.exists(), 'pid_path_exists': pid_path.exists(), 'log_path_exists': log_path.exists()}
if pid_path.exists():
    pid = int(pid_path.read_text(encoding='utf-8').strip())
    status['pid'] = pid
    status['proc_exists'] = Path(f'/proc/{pid}').exists()
if log_path.exists():
    text = log_path.read_text(encoding='utf-8', errors='replace')
    status['log_tail'] = text[-3000:]
print(json.dumps(status, ensure_ascii=False))


{"out_dir_exists": true, "pid_path_exists": true, "log_path_exists": true, "pid": 4701, "proc_exists": true, "log_tail": "[launcher] starting background training\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 6.5 MB/s eta 0:00:00\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 89.9 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 86.2 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.2 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 13.8 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.6 MB/s eta 0:00:00\n[colab] drive_root=/content/drive/MyDrive/thai-sign-translator\n[colab] repo_root=/content/thai-sign-translator\n[colab] data_roots=/content/kaggle/tsl51_v3,/content/kaggle/youtube_sl25_thai_v3\n[colab] out_dir=/content/checkpoints/pose_t5_v3_colab\n[colab] CUDA=True\n

In [ ]:
from __future__ import annotations
import json
from pathlib import Path

out_dir = Path('/content/checkpoints/pose_t5_v3_colab')
log_path = out_dir / 'train.log'
pid_path = out_dir / 'train.pid'
files = []
if out_dir.exists():
    for p in sorted(out_dir.iterdir()):
        if p.is_file():
            files.append({'name': p.name, 'size': p.stat().st_size})
status = {
    'pid': pid_path.read_text(encoding='utf-8').strip() if pid_path.exists() else None,
    'proc_exists': False,
    'files': files,
    'log_tail': '',
}
if status['pid']:
    status['proc_exists'] = Path(f"/proc/{status['pid']}").exists()
if log_path.exists():
    text = log_path.read_text(encoding='utf-8', errors='replace')
    status['log_tail'] = text[-5000:]
print(json.dumps(status, ensure_ascii=False))


{"pid": "4701", "proc_exists": true, "files": [{"name": "ckpt_step00000100.pt", "size": 3684199005}, {"name": "launch.json", "size": 844}, {"name": "train.log", "size": 2989}, {"name": "train.pid", "size": 4}], "log_tail": "[launcher] starting background training\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 6.5 MB/s eta 0:00:00\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 89.9 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 86.2 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.2 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 13.8 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.6 MB/s eta 0:00:00\n[colab] drive_root=/content/drive/MyDrive/thai-sign-translator\n[colab] repo_root=/content/thai-sign-translator\n[colab] data_roots=/content/kaggle/tsl51_v3,/content/k

In [ ]:
from __future__ import annotations
import json
from pathlib import Path

out_dir = Path('/content/checkpoints/pose_t5_v3_colab')
log_path = out_dir / 'train.log'
pid_path = out_dir / 'train.pid'
files = []
if out_dir.exists():
    for p in sorted(out_dir.iterdir()):
        if p.is_file():
            files.append({'name': p.name, 'size': p.stat().st_size})
status = {
    'pid': pid_path.read_text(encoding='utf-8').strip() if pid_path.exists() else None,
    'proc_exists': False,
    'files': files,
    'log_tail': '',
}
if status['pid']:
    status['proc_exists'] = Path(f"/proc/{status['pid']}").exists()
if log_path.exists():
    text = log_path.read_text(encoding='utf-8', errors='replace')
    status['log_tail'] = text[-5000:]
print(json.dumps(status, ensure_ascii=False))


{"pid": "4701", "proc_exists": true, "files": [{"name": "ckpt_step00000100.pt", "size": 3684199005}, {"name": "launch.json", "size": 844}, {"name": "train.log", "size": 2989}, {"name": "train.pid", "size": 4}], "log_tail": "[launcher] starting background training\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 6.5 MB/s eta 0:00:00\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 89.9 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 86.2 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.2 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 13.8 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.6 MB/s eta 0:00:00\n[colab] drive_root=/content/drive/MyDrive/thai-sign-translator\n[colab] repo_root=/content/thai-sign-translator\n[colab] data_roots=/content/kaggle/tsl51_v3,/content/k

In [ ]:
from __future__ import annotations
import json
from pathlib import Path

out_dir = Path('/content/checkpoints/pose_t5_v3_colab')
log_path = out_dir / 'train.log'
pid_path = out_dir / 'train.pid'
files = []
if out_dir.exists():
    for p in sorted(out_dir.iterdir()):
        if p.is_file():
            files.append({'name': p.name, 'size': p.stat().st_size})
status = {
    'pid': pid_path.read_text(encoding='utf-8').strip() if pid_path.exists() else None,
    'proc_exists': False,
    'files': files,
    'log_tail': '',
}
if status['pid']:
    status['proc_exists'] = Path(f"/proc/{status['pid']}").exists()
if log_path.exists():
    text = log_path.read_text(encoding='utf-8', errors='replace')
    status['log_tail'] = text[-5000:]
print(json.dumps(status, ensure_ascii=False))


{"pid": "4701", "proc_exists": true, "files": [{"name": "ckpt_step00000100.pt", "size": 3684199005}, {"name": "launch.json", "size": 844}, {"name": "tmp1dkbfhi4.tmp", "size": 2746601472}, {"name": "train.log", "size": 3062}, {"name": "train.pid", "size": 4}], "log_tail": "[launcher] starting background training\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 6.5 MB/s eta 0:00:00\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 89.9 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 86.2 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.2 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 13.8 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.6 MB/s eta 0:00:00\n[colab] drive_root=/content/drive/MyDrive/thai-sign-translator\n[colab] repo_root=/content/thai-sign-translator\n[cola

In [ ]:
from __future__ import annotations
import json
from pathlib import Path

out_dir = Path('/content/checkpoints/pose_t5_v3_colab')
log_path = out_dir / 'train.log'
pid_path = out_dir / 'train.pid'
files = []
if out_dir.exists():
    for p in sorted(out_dir.iterdir()):
        if p.is_file():
            files.append({'name': p.name, 'size': p.stat().st_size})
status = {
    'pid': pid_path.read_text(encoding='utf-8').strip() if pid_path.exists() else None,
    'proc_exists': False,
    'files': files,
    'log_tail': '',
}
if status['pid']:
    status['proc_exists'] = Path(f"/proc/{status['pid']}").exists()
if log_path.exists():
    text = log_path.read_text(encoding='utf-8', errors='replace')
    status['log_tail'] = text[-5000:]
print(json.dumps(status, ensure_ascii=False))


{"pid": "4701", "proc_exists": true, "files": [{"name": "ckpt_step00000100.pt", "size": 3684199005}, {"name": "ckpt_step00000200.pt", "size": 3684199005}, {"name": "ckpt_step00000300.pt", "size": 3684199005}, {"name": "ckpt_step00000400.pt", "size": 3684199005}, {"name": "launch.json", "size": 844}, {"name": "tmpyz_ch6yl.tmp", "size": 2867482624}, {"name": "train.log", "size": 3281}, {"name": "train.pid", "size": 4}], "log_tail": "[launcher] starting background training\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 6.5 MB/s eta 0:00:00\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 89.9 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 86.2 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.2 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 13.8 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from __future__ import annotations
import json
from pathlib import Path

out_dir = Path('/content/checkpoints/pose_t5_v3_colab')
log_path = out_dir / 'train.log'
pid_path = out_dir / 'train.pid'
files = []
if out_dir.exists():
    for p in sorted(out_dir.iterdir()):
        if p.is_file():
            files.append({'name': p.name, 'size': p.stat().st_size})
status = {
    'pid': pid_path.read_text(encoding='utf-8').strip() if pid_path.exists() else None,
    'proc_exists': False,
    'files': files,
    'log_tail': '',
}
if status['pid']:
    status['proc_exists'] = Path(f"/proc/{status['pid']}").exists()
if log_path.exists():
    text = log_path.read_text(encoding='utf-8', errors='replace')
    status['log_tail'] = text[-5000:]
print(json.dumps(status, ensure_ascii=False))


{"pid": "4701", "proc_exists": true, "files": [{"name": "ckpt_step00000100.pt", "size": 3684199005}, {"name": "ckpt_step00000200.pt", "size": 3684199005}, {"name": "ckpt_step00000300.pt", "size": 3684199005}, {"name": "ckpt_step00000400.pt", "size": 3684199005}, {"name": "ckpt_step00000500.pt", "size": 3684199005}, {"name": "launch.json", "size": 844}, {"name": "train.log", "size": 3281}, {"name": "train.pid", "size": 4}], "log_tail": "[launcher] starting background training\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 6.5 MB/s eta 0:00:00\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 89.9 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 86.2 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.2 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 13.8 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from __future__ import annotations
import json
from pathlib import Path

out_dir = Path('/content/checkpoints/pose_t5_v3_colab')
log_path = out_dir / 'train.log'
pid_path = out_dir / 'train.pid'
files = []
if out_dir.exists():
    for p in sorted(out_dir.iterdir()):
        if p.is_file():
            files.append({'name': p.name, 'size': p.stat().st_size})
status = {
    'pid': pid_path.read_text(encoding='utf-8').strip() if pid_path.exists() else None,
    'proc_exists': False,
    'files': files,
    'log_tail': '',
}
if status['pid']:
    status['proc_exists'] = Path(f"/proc/{status['pid']}").exists()
if log_path.exists():
    text = log_path.read_text(encoding='utf-8', errors='replace')
    status['log_tail'] = text[-5000:]
print(json.dumps(status, ensure_ascii=False))


{"pid": "4701", "proc_exists": true, "files": [{"name": "ckpt_step00000100.pt", "size": 3684199005}, {"name": "ckpt_step00000200.pt", "size": 3684199005}, {"name": "ckpt_step00000300.pt", "size": 3684199005}, {"name": "ckpt_step00000400.pt", "size": 3684199005}, {"name": "ckpt_step00000500.pt", "size": 3684199005}, {"name": "launch.json", "size": 844}, {"name": "train.log", "size": 3281}, {"name": "train.pid", "size": 4}], "log_tail": "[launcher] starting background training\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 6.5 MB/s eta 0:00:00\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 89.9 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 86.2 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.2 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 13.8 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from __future__ import annotations
import json
from pathlib import Path

out_dir = Path('/content/checkpoints/pose_t5_v3_colab')
log_path = out_dir / 'train.log'
pid_path = out_dir / 'train.pid'
files = []
if out_dir.exists():
    for p in sorted(out_dir.iterdir()):
        if p.is_file():
            files.append({'name': p.name, 'size': p.stat().st_size})
status = {
    'pid': pid_path.read_text(encoding='utf-8').strip() if pid_path.exists() else None,
    'proc_exists': False,
    'files': files,
    'log_tail': '',
}
if status['pid']:
    status['proc_exists'] = Path(f"/proc/{status['pid']}").exists()
if log_path.exists():
    text = log_path.read_text(encoding='utf-8', errors='replace')
    status['log_tail'] = text[-5000:]
print(json.dumps(status, ensure_ascii=False))


{"pid": "4701", "proc_exists": true, "files": [{"name": "ckpt_step00000300.pt", "size": 3684199005}, {"name": "ckpt_step00000400.pt", "size": 3684199005}, {"name": "ckpt_step00000500.pt", "size": 3684199005}, {"name": "launch.json", "size": 844}, {"name": "train.log", "size": 3281}, {"name": "train.pid", "size": 4}], "log_tail": "[launcher] starting background training\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 6.5 MB/s eta 0:00:00\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 89.9 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 86.2 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.2 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 13.8 MB/s eta 0:00:00\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.6 MB/s eta 0:00:00\n[colab] drive_root=/content/drive/MyDrive/thai-sign-transla